# End-to-end pipeline: data → model → evaluation → search

This notebook walks through the whole machine-learning side of the project in one place:

1. **Data** — check the downloaded images and your Label Studio labels
2. **Exploration** — what the catalogue and the labels look like
3. **Splits** — a reproducible train / validation / test split
4. **Embeddings** — what the pretrained EfficientNet-B0 encoder already "sees"
5. **Model** — frozen encoder + trainable decoder head for 10 attributes
6. **Training** — BCE loss, Adam, cosine learning-rate decay, early stopping
7. **Evaluation** — class-balanced F1, per-attribute F1 and a PR curve against an untrained baseline
8. **Use the model** — attribute predictions and visually similar items
9. **Save** — store the model for the API (and optionally upload it to S3)

**Before you start:** run `poe fetch_data` (downloads the public dataset) and label a first batch of images in Label Studio (see the README). The notebook needs at least a few dozen labeled images in `data/data/dataset.json`.

In [ ]:
%load_ext autoreload
%autoreload 2

# Run from the repository root so that data/ paths resolve
%cd ..

In [ ]:
import copy
import json
import random
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity
from torch import nn
from torch.utils.data import DataLoader

from minifigures_model.constants import DEFAULT_MODEL_TAG, get_data_folder, get_models_folder
from minifigures_model.dataset import MinifiguresDataset
from minifigures_model.metrics import (
    metric_class_balanced_f1,
    metric_f1_score,
    metric_precision,
    metric_recall,
)
from minifigures_model.model import EncoderDecoder
from minifigures_model.model_train import train
from minifigures_model.model_validate import validate

DATA_DIR = get_data_folder()
IMAGE_DIR = DATA_DIR / "minifigures"
SEED = 42
torch.manual_seed(SEED)

## 1. Data

`poe fetch_data` creates one PNG per image in `data/data/minifigures/`, a `catalog.json` with the character name and LEGO set of every image, and `dataset.json`, which maps each **labeled** image tag to its attributes. Labels come from Label Studio (`src/labeling/export_annotations.py`).

In [ ]:
catalog = json.loads((DATA_DIR / "catalog.json").read_text())
dataset = json.loads((DATA_DIR / "dataset.json").read_text())

print(f"Images in the catalogue: {len(catalog)}")
print(f"Labeled images:          {len(dataset)}")
assert len(dataset) >= 30, (
    "Label at least a few dozen images in Label Studio first (see the README)."
)

## 2. Exploration

How many photos are there per character, how often does each attribute occur, and how many attributes does an image typically have? Rare attributes are the hardest to learn — and exactly where active learning helps most.

In [ ]:
per_character = Counter(item["name"] for item in catalog.values())
plt.figure(figsize=(10, 4))
plt.bar(*zip(*per_character.most_common()))
plt.xticks(rotation=90)
plt.title("Images per character")
plt.tight_layout()
plt.show()

In [ ]:
label_counts = Counter(label for labels in dataset.values() for label in labels)
labels_per_image = Counter(len(labels) for labels in dataset.values())

_, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].barh(*zip(*sorted(label_counts.items(), key=lambda x: x[1])))
axs[0].set_title("Attribute frequency")
axs[1].bar(*zip(*sorted(labels_per_image.items())))
axs[1].set_title("Attributes per image")
axs[1].set_xlabel("number of attributes")
plt.tight_layout()
plt.show()

In [ ]:
# A few labeled examples
tags = random.Random(SEED).sample(sorted(dataset), k=min(8, len(dataset)))
_, axs = plt.subplots(2, 4, figsize=(14, 7))
for ax, tag in zip(axs.flat, tags):
    ax.imshow(Image.open(IMAGE_DIR / f"{tag}.png"))
    ax.set_title(f"{tag}\n{', '.join(dataset[tag])}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Train / validation / test split

A seeded shuffle keeps the split reproducible. The splits are written to `data/data/datasets/`, where `src/labeling/merge_labels_train_only.py` later adds new labels **to the training set only** — validation and test stay frozen, so models from different active-learning rounds are compared on the same images.

In [ ]:
split_dir = DATA_DIR / "datasets"
if (split_dir / "train.json").exists():
    splits = {
        name: json.loads((split_dir / f"{name}.json").read_text())
        for name in ("train", "val", "test")
    }
    print("Loaded the existing (frozen) splits")
else:
    keys = sorted(dataset)
    random.Random(SEED).shuffle(keys)
    n_val = n_test = max(1, int(0.15 * len(keys)))
    splits = {
        "val": {k: dataset[k] for k in keys[:n_val]},
        "test": {k: dataset[k] for k in keys[n_val : n_val + n_test]},
        "train": {k: dataset[k] for k in keys[n_val + n_test :]},
    }
    split_dir.mkdir(exist_ok=True)
    for name, data in splits.items():
        (split_dir / f"{name}.json").write_text(json.dumps(data, indent=2))
    print("Created new splits")

{name: len(data) for name, data in splits.items()}

In [ ]:
dataset_train = MinifiguresDataset(IMAGE_DIR, splits["train"])
dataset_val = MinifiguresDataset(IMAGE_DIR, splits["val"])
dataset_test = MinifiguresDataset(IMAGE_DIR, splits["test"])

# Every split must know the same attributes, in the same order
classes = dataset_train.classes
assert dataset_val.classes == classes and dataset_test.classes == classes, (
    "Each attribute must occur in train, val and test — label more images of the rare ones."
)

loader_train = DataLoader(dataset_train, batch_size=8, shuffle=True, drop_last=True)
loader_val = DataLoader(dataset_val, batch_size=8)
loader_test = DataLoader(dataset_test, batch_size=8)
classes

## 4. What the pretrained encoder already sees

EfficientNet-B0, pretrained on ImageNet, maps every image to a 1280-dimensional embedding. Even before any training, images that look alike get similar embeddings — which is what the **similar items** feature and the **active-learning** neighbour search are built on.

In [ ]:
model = EncoderDecoder(tag=DEFAULT_MODEL_TAG, classes=classes)
model.eval()

batch = next(iter(loader_train))
with torch.no_grad():
    embeddings = model.encoder(batch["image"]).numpy()
print(f"Batch {tuple(batch['image'].shape)} -> embeddings {embeddings.shape}")

similarity = cosine_similarity(embeddings)
plt.figure(figsize=(6, 5))
plt.imshow(similarity, vmin=0, vmax=1, cmap="viridis")
plt.xticks(range(len(batch["tag"])), batch["tag"], rotation=90, fontsize=8)
plt.yticks(range(len(batch["tag"])), batch["tag"], fontsize=8)
plt.colorbar(label="cosine similarity")
plt.title("Embedding similarity within one batch")
plt.tight_layout()
plt.show()

## 5. Model

Only the decoder head is trained: `1280 → 256 → 128 → 10` with ReLU and dropout, one logit per attribute. Freezing the encoder keeps training fast on a CPU and avoids overfitting a few hundred labels.

In [ ]:
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"Trainable parameters: {n_trainable:,}")
print(f"Frozen parameters:    {n_frozen:,}")

## 6. Training

Multi-label classification, so each attribute is an independent yes/no decision: `BCEWithLogitsLoss`. Adam with a cosine-decaying learning rate, and early stopping on validation F1 — the best weights are kept, not the last ones.

In [ ]:
n_epochs, patience = 40, 3
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam((p for p in model.parameters() if p.requires_grad), lr=3e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

history = {"train_loss": [], "train_f1": [], "val_loss": [], "val_f1": []}
best_f1, best_weights, best_epoch, waited = -1.0, None, 0, 0

for epoch in range(1, n_epochs + 1):
    losses, f1s = train(loader_train, model, optimizer, loss_fn, epoch)
    scheduler.step()
    metrics = validate(loader_val, model, loss_fn, epoch)

    history["train_loss"].append(float(np.mean(losses)))
    history["train_f1"].append(float(np.mean(f1s)))
    history["val_loss"].append(metrics["loss"])
    history["val_f1"].append(metrics["f1"])

    if metrics["f1"] > best_f1:
        best_f1, best_weights, best_epoch, waited = (
            metrics["f1"],
            copy.deepcopy(model.state_dict()),
            epoch,
            0,
        )
    else:
        waited += 1
        if waited >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

model.load_state_dict(best_weights)
print(f"Best epoch: {best_epoch}")

In [ ]:
_, axs = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axs, ("loss", "f1")):
    ax.plot(history[f"train_{metric}"], label="train")
    ax.plot(history[f"val_{metric}"], label="validation")
    ax.axvline(best_epoch - 1, color="gray", linestyle="--", label="best epoch")
    ax.set_title(metric.upper())
    ax.set_xlabel("epoch")
    ax.legend()
plt.tight_layout()
plt.show()

## 7. Evaluation on the test set

The test set was never used for training or early stopping. Class-balanced F1 averages the F1 of every attribute, so rare attributes count as much as common ones. An untrained model with the same architecture gives the baseline.

In [ ]:
def predict_all(model: EncoderDecoder, loader: DataLoader) -> tuple[torch.Tensor, torch.Tensor]:
    """Logits and targets for a whole data loader."""
    model.eval()
    logits, targets = [], []
    with torch.no_grad():
        for batch in loader:
            logits.append(model(batch["image"]))
            targets.append(batch["label"])
    return torch.cat(logits), torch.cat(targets)


def per_class_f1(logits: torch.Tensor, targets: torch.Tensor) -> dict[str, float]:
    """F1 of every attribute at a 0.5 probability threshold."""
    return {
        cls: metric_f1_score(
            pred=(logits[:, i] >= 0).double(), target=(targets[:, i] >= 0.5).double()
        )
        for i, cls in enumerate(classes)
    }


logits, targets = predict_all(model, loader_test)
baseline_logits, _ = predict_all(EncoderDecoder(tag="untrained", classes=classes), loader_test)

print(f"Test class-balanced F1 — trained:   {metric_class_balanced_f1(logits, targets):.3f}")
print(
    f"Test class-balanced F1 — untrained: {metric_class_balanced_f1(baseline_logits, targets):.3f}"
)

f1_cls = per_class_f1(logits, targets)
plt.figure(figsize=(7, 4))
plt.barh(*zip(*sorted(f1_cls.items(), key=lambda x: x[1])))
plt.xlim(0, 1)
plt.title("Test F1 per attribute")
plt.tight_layout()
plt.show()

In [ ]:
def pr_curve(logits: torch.Tensor, targets: torch.Tensor) -> tuple[list[float], list[float]]:
    """Macro-averaged precision and recall over probability thresholds 0.00 … 1.00."""
    probs = torch.sigmoid(logits)
    precision, recall = [], []
    for threshold in np.linspace(0, 1, 101):
        p, r = [], []
        for i in range(probs.shape[1]):
            pred = (probs[:, i] >= threshold).double()
            target = (targets[:, i] >= 0.5).double()
            p.append(metric_precision(pred=pred, target=target))
            r.append(metric_recall(pred=pred, target=target))
        precision.append(float(np.mean(p)))
        recall.append(float(np.mean(r)))
    return precision, recall


plt.figure(figsize=(5, 5))
for name, lg in (("trained", logits), ("untrained baseline", baseline_logits)):
    precision, recall = pr_curve(lg, targets)
    plt.plot(recall, precision, label=name)
plt.xlabel("recall")
plt.ylabel("precision")
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.legend()
plt.title("Precision-recall curve (test set)")
plt.tight_layout()
plt.show()

## 8. Use the model: attributes and similar items

The same two calls the API makes: `predict` returns a probability per attribute, and the encoder embeddings give the most similar catalogue images.

In [ ]:
query_tag = sorted(splits["test"])[0]
query = Image.open(IMAGE_DIR / f"{query_tag}.png")

prediction = model.predict(query)
print(
    f"{query_tag}: "
    + ", ".join(
        f"{k} {v:.2f}" for k, v in sorted(prediction.items(), key=lambda x: -x[1]) if v >= 0.5
    )
)

# Embed the whole catalogue once, then rank by cosine similarity
all_tags = sorted(p.stem for p in IMAGE_DIR.glob("*.png"))
catalog_embeddings = np.stack(
    [model.get_embedding(Image.open(IMAGE_DIR / f"{t}.png")).numpy() for t in all_tags]
)
scores = cosine_similarity(model.get_embedding(query).numpy()[None], catalog_embeddings)[0]
similar = [all_tags[i] for i in np.argsort(-scores) if all_tags[i] != query_tag][:5]

_, axs = plt.subplots(1, 6, figsize=(18, 3.5))
for ax, tag in zip(axs, [query_tag, *similar]):
    ax.imshow(Image.open(IMAGE_DIR / f"{tag}.png"))
    ax.set_title(("query: " if tag == query_tag else "") + tag, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 9. Save the model

The model is saved to `data/models/<tag>/` (weights and metadata), together with its training history. The API loads the model named by `MODEL_TAG` (default `my_model`). To deploy it, set `MODELS_BUCKET` to the S3 bucket created by Terraform.

In [ ]:
model.save()
model_dir = get_models_folder() / DEFAULT_MODEL_TAG
(model_dir / "history.json").write_text(
    json.dumps({**history, "best_epoch": best_epoch, "n_labeled": len(dataset)}, indent=2)
)
print(f"Saved to {model_dir}")

In [ ]:
# Optional: upload the model so the deploy pipeline can serve it
import os
import subprocess

bucket = os.environ.get("MODELS_BUCKET")
if bucket:
    subprocess.run(
        [
            "aws",
            "s3",
            "cp",
            str(model_dir),
            f"s3://{bucket}/models/{DEFAULT_MODEL_TAG}",
            "--recursive",
        ],
        check=True,
    )
else:
    print("MODELS_BUCKET is not set — skipping the S3 upload")